In [1]:
from tqdm.notebook import tqdm

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
# check if cuda is available
if torch.cuda.is_available():
    device = "cuda"
    torch.cuda.empty_cache()
else:
    device = "cpu"

### Tiny LLM built on Pan Tadeusz or Odyssey

In [ ]:
# get training text

In [4]:
!wget https://wolnelektury.pl/media/book/txt/pan-tadeusz.txt

--2025-05-15 11:08:38--  https://wolnelektury.pl/media/book/txt/pan-tadeusz.txt
Resolving wolnelektury.pl (wolnelektury.pl)... 51.83.143.148, 2001:41d0:602:3294::
Connecting to wolnelektury.pl (wolnelektury.pl)|51.83.143.148|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 493758 (482K) [text/plain]
Saving to: ‘pan-tadeusz.txt’

pan-tadeusz.txt     100%[===================>] 482.19K   472KB/s    in 1.0s    

2025-05-15 11:08:41 (472 KB/s) - ‘pan-tadeusz.txt’ saved [493758/493758]



In [ ]:
#!wget https://classics.mit.edu/Homer/odyssey.mb.txt

In [5]:
lines = []
with open("pan-tadeusz.txt") as f:
    for l in f:
        lines.append(l.strip())

# devide into stanzas
stanzas = []
i_prev = 0
for i in range(1, len(lines)):
    if lines[i] == "":
        stanzas.append(" ".join(lines[i_prev:i]))
        i_prev = i + 1

for z in stanzas[20:30]:
    print(z)

«Dobrze mój Tadeuszu, (bo tak nazywano Młodzieńca, który nosił Kościuszkowskie miano Na pamiątkę, że w czasie wojny się urodził) Dobrze mój Tadeuszu, żeś się dziś nagodził Do domu, właśnie kiedy mamy panien wiele. Stryjaszek myśli wkrótce sprawić ci wesele; Jest z czego wybrać; u nas towarzystwo liczne Od dni kilku zbiera się na sądy graniczne, Dla skończenia dawnego z panem Hrabią sporu. I pan Hrabia ma jutro sam zjechać do dworu; Podkomorzy już zjechał z żoną i z córkami. Młodzież poszła do lasu bawić się strzelbami, A starzy i kobiety żniwo oglądają Pod lasem i tam pewnie na młodzież czekają. Pójdziemy, jeśli zechcesz, i wkrótce spotkamy Stryjaszka, Podkomorstwo i szanowne damy».
Pan Wojski z Tadeuszem idą pod las drogą, I jeszcze się do woli nagadać nie mogą.
Słońce ostatnich kresów nieba dochodziło, Mniej silnie, ale szerzej niż we dnie świeciło, Całe zaczerwienione, jak zdrowe oblicze Gospodarza, gdy prace skończywszy rolnicze Na spoczynek powraca. Już krąg promienisty Spuszcza s

In [6]:
# stats
print(max(len(z) for z in stanzas))
print(sum(len(z) for z in stanzas)/len(stanzas))
words = set()
for z in stanzas:
    for w in z.split():
        words.add(w)
print(len(words))

3881
497.50958286358514
27622


In [ ]:
# prepare data

In [7]:
# add START/END symbols
x_txt = ["START " + z + " END" for z in stanzas]

In [8]:
# torchtext is deprecated
import keras

In [9]:
tv = keras.layers.TextVectorization(output_sequence_length=300)
tv.adapt(x_txt)

In [10]:
vocab = tv.get_vocabulary()
print(len(vocab))
vocab[:10]

19563


['',
 '[UNK]',
 np.str_('i'),
 np.str_('w'),
 np.str_('się'),
 np.str_('z'),
 np.str_('na'),
 np.str_('nie'),
 np.str_('start'),
 np.str_('end')]

In [11]:
# inverse vocab:
iw = dict()
for i, w in enumerate(vocab):
    iw[w] = i

In [12]:
xp = tv(x_txt)
xp[:10]

<tf.Tensor: shape=(10, 300), dtype=int64, numpy=
array([[    8,  6946,  1875, ...,     0,     0,     0],
       [    8,    27,    53, ...,     0,     0,     0],
       [    8, 17286, 19559, ...,     0,     0,     0],
       ...,
       [    8,   515,  2403, ...,     0,     0,     0],
       [    8,     9,     0, ...,     0,     0,     0],
       [    8,     9,     0, ...,     0,     0,     0]])>

In [13]:
xp = torch.tensor(xp.numpy())

In [14]:
y = xp[:,1:] # shift left: predict next word
x = xp[:,:-1]

In [15]:
print(x[0][:10])
print(y[0][:10])

tensor([   8, 6946, 1875,    9,    0,    0,    0,    0,    0,    0])
tensor([6946, 1875,    9,    0,    0,    0,    0,    0,    0,    0])


In [16]:
from torch.utils.data import DataLoader, TensorDataset

In [17]:
d_train = TensorDataset(torch.tensor(x), torch.tensor(y))
dl_train = DataLoader(d_train, batch_size=16, shuffle=True)

<ipython-input-17-62a7cd076784>:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  d_train = TensorDataset(torch.tensor(x), torch.tensor(y))


### Transformer elements

In [19]:
import numpy as np

In [20]:
def positional_encoding(length, depth):
  depth = depth/2
  positions = np.arange(length)[:, np.newaxis]
  depths = np.arange(depth)[np.newaxis, :]/depth
  pos_enc_complex = np.exp(1j*positions/(10000**depths))
  pos_enc = pos_enc_complex.view(np.float64)
  return pos_enc

In [21]:
class PositionBlock(nn.Module):
    def __init__(self, embed_dim, seq_length):
        super().__init__()
        pe = positional_encoding(length=seq_length, depth=embed_dim)
        pe = pe.astype(np.float32)[np.newaxis,...]
        self.pe = torch.tensor(pe, requires_grad=False).to(device)
    def forward(self, x):
        return x + self.pe

In [22]:
class TransformerBlock(nn.Module):
    def __init__(self, num_heads, embed_dim, dropout_rate=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.dropout_rate = dropout_rate
        self.mha = nn.MultiheadAttention(num_heads=num_heads,
                                         embed_dim=embed_dim,
                                         batch_first=True)
        self.FF = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.dp = nn.Dropout(self.dropout_rate)

    def forward(self, inputs, mask):
        a, _ = self.mha(query=inputs, value=inputs, key=inputs,
                        need_weights=False,
                        is_causal=True,key_padding_mask=mask,
                        attn_mask=(nn.Transformer.generate_square_subsequent_mask(299, device=device) < -100))
        e2 = self.ln1(a + inputs)
        # feed-forward part
        e3 = self.FF(e2)
        e4 = self.ln2(e3 + e2)
        e4 = self.dp(e4)
        return e4

In [23]:
class LLM(nn.Module):
    def __init__(self, d=64):
        super().__init__()
        self.d = d
        self.emb = nn.Embedding(len(vocab), d)
        self.pos = PositionBlock(d, 299)
        self.t1 = TransformerBlock(2, d)
        self.t2 = TransformerBlock(2, d)
        self.fl = nn.Flatten()
        self.l1 = nn.LazyLinear(len(vocab))
    def forward(self, x):
        mask = (x == 0).to(device)
        #mask = torch.where(mask, -torch.inf, 0)
        emb = self.emb(x)
        pos = self.pos(emb)
        t1 = self.t1(pos, mask)
        t2 = self.t2(t1, mask)
        #x = self.fl(t2)
        x = self.l1(t2)
        return x

In [24]:
# training loop
import torch.optim as optim

def fit(net, train=dl_train, epochs=10,
        learning_rate=None):
    net = net.to(device)

    if learning_rate is None:
        optimizer = optim.Adam(net.parameters())
    else:
        optimizer = optim.Adam(net.parameters(), lr=learning_rate)
    #loss = torch.nn.CrossEntropyLoss()
    loss = torch.nn.CrossEntropyLoss(ignore_index=0)  # ignore loss on padding
    for e in range(epochs):
        net.train()
        epoch_loss = 0
        for X, y in tqdm(train):
            optimizer.zero_grad()
            X = X.to(device)
            y = y.to(device)
            pred = net(X)
            l = loss(pred.view(-1, len(vocab)), y.view(-1))
            epoch_loss += l.item()
            l.backward()
            optimizer.step()
        print("Epoch", e, "loss:", epoch_loss/len(train))

In [25]:
llm = LLM()
fit(llm)

  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 0 loss: 9.140469176428658


  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 1 loss: 8.256247196878705


  0%|          | 0/56 [00:00<?, ?it/s]

...

Epoch 7 loss: 7.5207679271698


  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 8 loss: 7.30639397246497


  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 9 loss: 7.0745958515575955


In [26]:
fit(llm, epochs=10)

  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 0 loss: 6.993986998285566


  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 1 loss: 6.753488727978298


  0%|          | 0/56 [00:00<?, ?it/s]

...

Epoch 7 loss: 5.5181567413466315


  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 8 loss: 5.309704465525491


  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 9 loss: 5.1098217112677435


In [27]:
fit(llm, epochs=100)

  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 0 loss: 5.219888303961072


  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 1 loss: 5.036574065685272


  0%|          | 0/56 [00:00<?, ?it/s]

...

Epoch 97 loss: 1.0080400194440569


  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 98 loss: 0.9908595191580909


  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 99 loss: 0.9772503322788647


In [30]:
fit(llm, epochs=100, learning_rate=0.0001)

  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 0 loss: 0.9244016313127109


  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 1 loss: 0.9020861461758614


  0%|          | 0/56 [00:00<?, ?it/s]

...

Epoch 97 loss: 0.7548062662993159


  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 98 loss: 0.7558859001312938


  0%|          | 0/56 [00:00<?, ?it/s]

Epoch 99 loss: 0.7439455028091159


In [31]:
x_t = torch.zeros(x.shape[1], dtype=int, device=device)
x_t[0] = iw["start"];t0=0 # start
x_t[1] = iw["tadeusz"];t0=1
x_t[2] = iw["i"];t0=2
x_t[3] = iw["gerwazy"];t0=3
llm.eval()
for t in range(t0, 30):
    #print(llm(x_t.view(1,-1)).argmax(axis=2).shape)
    x_t[t+1] = llm(x_t.view(1,-1)).argmax(axis=2)[0,t]
    for w in x_t:
        print(vocab[w], end=" ")
    print()

start tadeusz i gerwazy porządkują                                                                                                                                                                                                                                                                                                       
start tadeusz i gerwazy porządkują rozdają                                                                                                                                                                                                                                                                                                      
start tadeusz i gerwazy porządkują rozdają oręże                                                                                                                                                                                                                                                                                             

In [32]:
x_t = torch.zeros(x.shape[1], dtype=int, device=device)
x_t[0] = iw["start"];t0=0 # start
x_t[1] = iw["tadeusz"];t0=1
#x_t[2] = iw["i"];t0=2
#x_t[3] = iw["wojski"];t0=3
llm.eval()
for t in range(t0, 20):
    logits = llm(x_t.view(1,-1))[0][t]
    logits /= 0.9 # temperature
    probs = F.softmax(logits, dim=-1)
    probs = probs.detach().cpu().numpy().astype(float)
    print(probs.sum())
    probs = probs/probs.sum()
    t_pred = np.flatnonzero(np.random.multinomial(1, probs)).item()
    x_t[t+1] = t_pred
    for w in x_t:
        print(vocab[w], end=" ")
    print()

1.000000011483804
start tadeusz spał                                                                                                                                                                                                                                                                                                         
0.999999928506336
start tadeusz spał pod                                                                                                                                                                                                                                                                                                        
0.9999999753022568
start tadeusz spał pod oknem                                                                                                                                                                                                                                                                                          